# SpOC 2: Quantum Communications Constellations
Projekt për lëndën Dizajni dhe Analiza e Algoritmeve.

## Problemi (shkurt)
Dy konstelacione satelitore të tipit Walker rreth planetit "New Mars" ndërlidhin 7 anije mëmë në orbitë me 4 rovera në sipërfaqe. Dy objektiva bien ndesh:
- **J1**: kostoja mesatare e rrugës më të shkurtër të komunikimit (mbi 10 vjet dhe mbi të gjitha çiftet anije-rover). Sa më e ulët, aq më mirë.
- **J2**: kostoja e satelitëve (varet nga numri i tyre dhe nga cilësia η). Sa më e ulët, aq më mirë.

Kërkohet fronti i Pareto-s, i vlerësuar me hipervëllim (pika referuese [1.2, 1.4]). Vektori i vendimit ka 20 variabla (10 të vazhdueshme dhe 10 të plota) dhe dy kufizime: roverat të jenë të paktën 3000 km larg njëri-tjetrit, dhe satelitët të paktën 50 km larg.

## Të dhënat (dataset)
Dataset-i `spoc2data` përmban dy skedarë nga repo zyrtare e ESA-s (https://github.com/esa/spoc2), nga folderi "Quantum Communications Constellations":
- `constellations_udp.py`: kodi i problemit (llogarit J1, J2 dhe kufizimet)
- `rovers.txt`: 100 pozicionet e mundshme të roverave (gjerësia dhe gjatësia në radianë)

Skedarët në `/kaggle/input` janë vetëm për lexim, prandaj i kopjojmë te `/kaggle/working`.

## Setup i ambientit
Kodi i ESA-s kërkon pykep 2.6. Kaggle ka Python 3.13, ku ky version nuk instalohet. Prandaj, me micromamba krijojmë një ambient me Python 3.10 dhe pykep 2.6.4, dhe e ekzekutojmë kodin me atë Python. Skedari `rovers.txt` vendoset te `data/spoc2/constellations/`, ku e kërkon kodi.

## Rezultatet deri tani
- Ambienti funksionon dhe pykep del versioni 2.6.
- `udp.example()` përfundon për 0.0 sekonda, ndaj `fitness` u mat veçmas me një vektor të vendosur me dorë.
- Vektori i provës (të dy konstelacionet me S=6, P=4, η=100) dha **J1 = 0.948** dhe **J2 = 0.048**. Të dy kufizimet dolën negative, pra zgjidhja është e vlefshme.
- Një vlerësim `fitness` zgjat rreth **0.73 sekonda** për këtë madhësi konstelacioni.
- Kufijtë e variablave morën nga `get_bounds()` dhe përputhen me faqen e ESA-s.

## Hapat e radhës
1. Matja e kohës së vlerësimit për konstelacione të ndryshme në madhësi.
2. Filtri i roverave (kufizimi 3000 km).
3. Kërkim i rastësishëm si bazë dhe llogaritja e hipervëllimit.
4. Algoritëm më i mirë (evolucionar ose kërkim lokal) dhe krahasim me bazën.
5. Grafiku J1 kundrejt J2, përfundimet dhe raporti.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!ls -R /kaggle/input

/kaggle/input:
datasets

/kaggle/input/datasets:
mirenahaliti

/kaggle/input/datasets/mirenahaliti:
spoc2data

/kaggle/input/datasets/mirenahaliti/spoc2data:
constellations_udp.py  rovers.txt


In [2]:
!grep -n "pk\." /kaggle/input/datasets/mirenahaliti/spoc2data/constellations_udp.py

76:        jd0, fr = pk.epoch(self._t0, 'mjd2000').jd, 0.0 # reference Julian date
81:        self.ep_ref = pk.epoch_from_iso_string("19491231T000000")
98:        self.LOS = 1.05 * pk.EARTH_RADIUS / 1000.0
100:        self.R_p = pk.EARTH_RADIUS / 1000.0
165:        d = scipy.spatial.distance.cdist(pos, pos, lambda u, v: pk.EARTH_RADIUS/1000 * safe_arccos(u, v))
295:        mean_motion = np.sqrt(pk.MU_EARTH/a**3/pk.EARTH_RADIUS**3)
567:        mean_motion = np.sqrt(pk.MU_EARTH / sma**3 / pk.EARTH_RADIUS**3) * 24 * 60 * 60 / (2 * np.pi)
692:        r = pk.EARTH_RADIUS/1000


In [3]:
!mkdir -p /kaggle/working/mm && cd /kaggle/working/mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

bin/micromamba


In [4]:
!/kaggle/working/mm/bin/micromamba create -y -n spoc2 -r /kaggle/working/mm/root -c conda-forge python=3.10 "pykep<3" numpy scipy networkx sgp4 matplotlib seaborn pygmo 2>&1 | tail -n 15

Linking seaborn-base-0.13.2-pyhd8ed1ab_3
Linking statsmodels-0.15.0-np2py310h83c7da3_1
Linking seaborn-0.13.2-hd8ed1ab_3

Transaction finished


To activate this environment, use:

    micromamba activate spoc2

Or to execute a single command in this environment, use:

    micromamba run -n spoc2 mycommand



In [5]:
!/kaggle/working/mm/root/envs/spoc2/bin/python -c "import pykep; print(pykep.__version__)"

{'major': 2, 'minor': 6}


In [6]:
import os, shutil

src = "/kaggle/input/datasets/mirenahaliti/spoc2data"
os.makedirs("/kaggle/working/data/spoc2/constellations", exist_ok=True)
shutil.copy(src + "/rovers.txt", "/kaggle/working/data/spoc2/constellations/rovers.txt")
shutil.copy(src + "/constellations_udp.py", "/kaggle/working/constellations_udp.py")

code = '''import time
from constellations_udp import constellation_udp

udp = constellation_udp()
t0 = time.time()
udp.example()
print("Koha totale: %.1f sekonda" % (time.time() - t0))
'''
open("/kaggle/working/test1.py", "w").write(code)

print(os.listdir("/kaggle/working"))

['.virtual_documents', 'constellations_udp.py', 'test1.py', 'mm', 'data']


In [7]:
!cd /kaggle/working && MPLBACKEND=Agg /kaggle/working/mm/root/envs/spoc2/bin/python test1.py

Koha totale: 0.0 sekonda


In [8]:
!sed -n '516,553p' /kaggle/working/constellations_udp.py


    def example(self, verbose=False):
        """A random chromosome example for the constellation optimization

        Returns:
            list: a valid chromosome representing a possible constellation design
        """
        if verbose:
            print("CHROMOSOME:")
            print("x = [a1, e1, i1, w1, eta1] + [a2, e2, i2, w2, eta2] + [S1, P1, F1] + [S2, P2, F2] + [r1, r2, r3, r4]")
            print(100 * "-")
            print("a1: float representing the normalized semi-major axis of Walker1 satellite orbits (in km) [1.05,1.8]")
            print("e1: float representing the eccentricity [0, 0.1]")
            print("i1: float representing the inclination [0, pi]")
            print("w1: float representing the argument of the perigee [0, 2pi]")
            print("eta1: float defined as the quality indicator of satellites in the first walker constellation [0, 100]")
            print(100 * "-")
            print("a2: float representing the normalized semi-major axis of Wal

In [9]:
code = '''import time
from constellations_udp import constellation_udp

udp = constellation_udp()

lb, ub = udp.get_bounds()
print("Numri i variablave:", len(lb))
print("Kufijte e poshtem:", list(lb))
print("Kufijte e siperm:", list(ub))

x = [1.3, 0.01, 1.0, 0.0, 100.0,
     2.5, 0.05, 1.0, 0.0, 100.0,
     6, 4, 1, 6, 4, 1,
     0, 10, 20, 30]

t0 = time.time()
f = udp.fitness(x)
dt = time.time() - t0
print("fitness =", f)
print("Koha e nje fitness: %.2f sekonda" % dt)
'''
open("/kaggle/working/test2.py", "w").write(code)

466

In [10]:
!cd /kaggle/working && MPLBACKEND=Agg /kaggle/working/mm/root/envs/spoc2/bin/python test2.py

Numri i variablave: 20
Kufijte e poshtem: [1.06, 0.0, 0.0, 0.0, 1.0, 2.0, 0.0, 0.0, 0.0, 1.0, 4, 2, 0, 4, 2, 0, 0, 0, 0, 0]
Kufijte e siperm: [1.8, 0.02, 3.141592653589793, 6.283185307179586, 1000.0, 3.5, 0.1, 3.141592653589793, 6.283185307179586, 1000.0, 10, 10, 9, 10, 10, 9, 99, 99, 99, 99]
fitness = [0.9483319085696353, 0.048, np.float64(-3677.037557864227), np.float64(-243.7086752347144)]
Koha e nje fitness: 0.73 sekonda


In [11]:
code = '''import time
from constellations_udp import constellation_udp

udp = constellation_udp()

# [a1,e1,i1,w1,eta1, a2,e2,i2,w2,eta2, S1,P1,F1, S2,P2,F2, r1,r2,r3,r4]
teste = {
    "i vogel (S=4, P=2, eta=1)":       [1.3,0.01,1.0,0.0,1.0,    2.5,0.05,1.0,0.0,1.0,    4,2,1,   4,2,1,   0,10,20,30],
    "i mesem (S=6, P=4, eta=100)":     [1.3,0.01,1.0,0.0,100.0,  2.5,0.05,1.0,0.0,100.0,  6,4,1,   6,4,1,   0,10,20,30],
    "i madh (S=10, P=10, eta=1000)":   [1.3,0.01,1.0,0.0,1000.0, 2.5,0.05,1.0,0.0,1000.0, 10,10,1, 10,10,1, 0,10,20,30],
}

for emri, x in teste.items():
    print(emri)
    try:
        t0 = time.time()
        f = udp.fitness(x)
        dt = time.time() - t0
        print("   J1 = %.4f   J2 = %.5f" % (f[0], f[1]))
        print("   kufizimi rovera = %.1f   kufizimi satelita = %.1f" % (float(f[2]), float(f[3])))
        print("   koha = %.2f sekonda" % dt)
    except Exception as e:
        print("   GABIM:", e)
'''
open("/kaggle/working/test3.py", "w").write(code)

!cd /kaggle/working && MPLBACKEND=Agg /kaggle/working/mm/root/envs/spoc2/bin/python test3.py

i vogel (S=4, P=2, eta=1)
   J1 = 15.9988   J2 = 0.00016
   kufizimi rovera = -3677.0   kufizimi satelita = -1027.5
   koha = 0.13 sekonda
i mesem (S=6, P=4, eta=100)
   J1 = 0.9483   J2 = 0.04800
   kufizimi rovera = -3677.0   kufizimi satelita = -243.7
   koha = 0.69 sekonda
i madh (S=10, P=10, eta=1000)
   J1 = 0.7421   J2 = 2.00000
   kufizimi rovera = -3677.0   kufizimi satelita = -516.3
   koha = 9.29 sekonda
